# Create Health Research Board Ireland Awards

Creates OpenAlex award rows from Health Research Board (HRB) Ireland's official Grants approved database.

**Data source:** `https://www.hrb.ie/funding-category/research-funding/investments-impacts/grants-approved/` plus HRB's public WordPress REST endpoint `https://www.hrb.ie/wp-json/wp/v2/grant-approved`. REST gives all grant posts; rendered detail pages provide structured sidebar fields for award year, duration, grant value, principal investigator, host institution, scheme, scheme type, and HRB broad research area.
**S3 location:** `s3a://openalex-ingest/awards/hrb_ireland/hrb_ireland_projects.parquet`
**OpenAlex funder:** `F4320312041` - Health Research Board (ROR `https://ror.org/003hb2249`, DOI `10.13039/100010414`), country `IE`.
**Provenance:** `hrb_grants_approved`.
**Priority:** 218.

**Schema choices:**
- One row per unique HRB WordPress grant post. The REST endpoint returned 2,155 rows, including 65 exact duplicate post IDs across page boundaries; the script drops those exact duplicates and emits 2,090 unique grant awards.
- HRB does not publish a grant-number field on these pages, so `funder_award_id = hrb-{wp_post_id}` from the official source. Slug and landing page are retained for auditability.
- `amount` from `Grant Value` and `currency = EUR`; blanks/failed detail pages remain NULL.
- `lead_investigator` from the published Principal Investigator and Host Institution. Host affiliation country is NULL because the source does not publish per-award country.
- HRB publishes award year and duration but not day-level start/end dates. `start_year` is populated from Award Year; `start_date`, `end_date`, and `end_year` remain NULL.
- `description` comes from the grant post content/abstract when published.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrb_ireland_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hrb_ireland/hrb_ireland_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS total_hrb_ireland_grants FROM openalex.awards.hrb_ireland_raw;


## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns: `funder_award_id`, `wp_post_id`, `slug`, `display_name`, `description`, `award_year_raw`, `duration_raw`, `duration_months`, `amount`, `currency`, `principal_investigator_raw`, `lead_given_name`, `lead_family_name`, `host_institution`, `funder_scheme`, `scheme_type`, `hrb_broad_research_area`, `funding_type`, `start_year`, `end_year`, `landing_page_url`, `wp_date`, `wp_modified`, `source_page_url`, `provenance`, `funder_id`, `funder_display_name`, `downloaded_at`.


In [ ]:
%sql
DESCRIBE openalex.awards.hrb_ireland_raw;


In [ ]:
%sql
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'hrb_ireland_raw'
  AND LOWER(column_name) RLIKE 'amount|value|currency|duration|year'
ORDER BY column_name;


In [ ]:
%sql
SELECT funder_award_id, display_name, principal_investigator_raw, host_institution,
       amount, currency, start_year, duration_raw, funder_scheme, landing_page_url
FROM openalex.awards.hrb_ireland_raw
LIMIT 10;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(principal_investigator_raw) AS has_pi,
    COUNT(lead_family_name) AS has_lead_family,
    COUNT(host_institution) AS has_host,
    COUNT(TRY_CAST(amount AS DOUBLE)) AS has_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    COUNT(TRY_CAST(start_year AS INT)) AS has_start_year,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.hrb_ireland_raw;


In [ ]:
%sql
SELECT funding_type, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS eur_total
FROM openalex.awards.hrb_ireland_raw
GROUP BY funding_type
ORDER BY awards DESC;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS eur_total
FROM openalex.awards.hrb_ireland_raw
GROUP BY funder_scheme
ORDER BY awards DESC
LIMIT 25;


## Step 1.6: Fail-Fast - Verify HRB Funder Exists

F4320312041 is a Crossref-registered `F4320*` funder, so it should exist in `openalex.common.funder`. This assertion prevents a silent zero-row transform.


In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Missing Health Research Board funder F4320312041 in openalex.common.funder'
) AS hrb_ireland_funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320312041;


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320312041;


## Step 2: Transform to OpenAlex Awards Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrb_ireland_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT
        4320312041 AS funder_id,
        COALESCE(MAX(display_name), 'Health Research Board') AS display_name,
        COALESCE(MAX(ror_id), 'https://ror.org/003hb2249') AS ror_id,
        COALESCE(MAX(doi), '10.13039/100010414') AS doi
    FROM openalex.common.funder
    WHERE funder_id = 4320312041
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(start_year AS INT) END AS start_year_int,
        NULLIF(TRIM(display_name), '') AS display_name_norm,
        NULLIF(TRIM(description), '') AS description_norm,
        NULLIF(TRIM(host_institution), '') AS leader_affiliation_name,
        NULLIF(TRIM(lead_given_name), '') AS leader_given_name,
        NULLIF(TRIM(lead_family_name), '') AS leader_family_name,
        NULLIF(TRIM(funder_scheme), '') AS funder_scheme_norm,
        NULLIF(TRIM(funding_type), '') AS funding_type_norm
    FROM openalex.awards.hrb_ireland_raw
)
SELECT
    abs(xxhash64(CONCAT('4320312041:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name_norm AS display_name,
    rp.description_norm AS description,
    4320312041 AS funder_id,
    rp.funder_award_id,
    CASE WHEN rp.amount_double > 0 THEN rp.amount_double ELSE NULL END AS amount,
    CASE WHEN rp.amount_double > 0 THEN 'EUR' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    COALESCE(rp.funding_type_norm, 'research') AS funding_type,
    rp.funder_scheme_norm AS funder_scheme,
    'hrb_grants_approved' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    CASE
        WHEN rp.start_year_int > YEAR(current_date()) + 1 THEN NULL
        ELSE rp.start_year_int
    END AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN rp.leader_given_name IS NOT NULL OR rp.leader_family_name IS NOT NULL OR rp.leader_affiliation_name IS NOT NULL THEN
            struct(
                rp.leader_given_name AS given_name,
                rp.leader_family_name AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.leader_affiliation_name AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320312041:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
CROSS JOIN funder_resolved f
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name_norm IS NOT NULL;


## Step 3: Insert into `openalex_awards_raw` at Priority 218


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hrb_grants_approved' AND priority = 218;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    218 AS priority
FROM openalex.awards.hrb_ireland_awards;


## Step 6: Verification


In [ ]:
%sql
SELECT COUNT(*) AS total_hrb_ireland_awards FROM openalex.awards.hrb_ireland_awards;


In [ ]:
%sql
-- Duplicate funder_award_id / id must be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.hrb_ireland_awards;


In [ ]:
%sql
-- Completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_affiliation,
    SUM(CASE WHEN lead_investigator.affiliation.country IS NOT NULL THEN 1 ELSE 0 END) AS has_affiliation_country,
    COUNT(start_year) AS has_start_year,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_lead_name,
    collect_set(currency) AS currencies
FROM openalex.awards.hrb_ireland_awards;


In [ ]:
%sql
-- Amount coverage (not waived: HRB publishes Grant Value).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    ROUND(AVG(amount), 2) AS avg_amount,
    percentile_approx(amount, 0.5) AS median_amount
FROM openalex.awards.hrb_ireland_awards;


In [ ]:
%sql
-- PI frequency (Step 6.4a): no name should dominate; no institution-as-name.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.hrb_ireland_awards
GROUP BY 1, 2
ORDER BY n DESC
LIMIT 25;


In [ ]:
%sql
SELECT funding_type, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS eur_total
FROM openalex.awards.hrb_ireland_awards
GROUP BY funding_type
ORDER BY awards DESC;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS eur_total
FROM openalex.awards.hrb_ireland_awards
GROUP BY funder_scheme
ORDER BY awards DESC
LIMIT 25;


In [ ]:
%sql
SELECT id, funder_award_id, SUBSTRING(display_name, 1, 90) AS title,
       amount, currency, start_year,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 70) AS host
FROM openalex.awards.hrb_ireland_awards
ORDER BY start_year DESC, funder_award_id
LIMIT 20;


In [ ]:
%sql
-- Confirm the priority insert landed.
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hrb_grants_approved'
GROUP BY priority, provenance;


## Handoff / Admin Notes

Contractor validation completed locally with `scripts/local/hrb_ireland_to_s3.py --skip-upload`: 2,090 unique HRB grant awards after dropping 65 exact duplicate REST rows by WordPress post ID; 99.3% description coverage; 99.9% PI coverage; 99.7% host coverage; 99.95% positive EUR amount coverage; 99.9% start-year coverage. Contractor has no S3/Databricks access, so admin must upload the parquet, run this notebook, and perform Databricks/API QA.
